In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()
    
print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
CACHE_DIR = BASE_DIR / "LSTM_cache_TL/Transformer_exp_IT_AT"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for LSTM. 

In [ ]:
ALL_REGIONS = [
    'CH',
    'NOR',
    'ISL',
    'FR',
]
TARGET_REGIONS = ['IT_AT']

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
}

run_flag_by_code = {
    'CH': False,
    'NOR': False,
    'ISL': False,
    'FR': False,
    'SJM': False,
    "IT_AT": False,
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=ALL_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

res_xreg_by_source = {
    region: monthly_cache[region]
    for region in ALL_REGIONS + TARGET_REGIONS
}

In [ ]:
AT_GLACIERS = [
    'GOLDBERG K.',
    'HALLSTAETTER G.',
    'HINTEREIS F.',
    'JAMTAL F.',
    'KESSELWAND F.',
    'KLEINFLEISS K.',
    'OE. WURTEN K.',
    'VENEDIGER K.',
    'VERNAGT F.',
    'ZETTALUNITZ/MULLWITZ K.',
]

IT_GLACIERS = [
    'CAMPO SETT.',
    'CARESER',
    'CARESER CENTRALE',
    'CARESER OCCIDENTALE',
    'CARESER ORIENTALE',
    'CIARDONEY',
    'FONTANA BIANCA / WEISSBRUNNF.',
    'GRAND ETRET',
    'LUNGA (VEDRETTA) / LANGENF.',
    'LUPO',
    'MALAVALLE (VEDR. DI) / UEBELTALF.',
    'PENDENTE (VEDR.) / HANGENDERF.',
    'RIES OCC. (VEDR. DI) / RIESERF. WESTL.',
    'SURETTA MERIDIONALE',
]

data_AT = monthly_cache['IT_AT']['data_monthly'][monthly_cache['IT_AT'][
    'data_monthly']['GLACIER'].isin(AT_GLACIERS)].copy()

data_IT = monthly_cache['IT_AT']['data_monthly'][monthly_cache['IT_AT'][
    'data_monthly']['GLACIER'].isin(IT_GLACIERS)].copy()

### Finetuning glaciers:

#### Automatic picking:

In [ ]:
CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']

# remap cache keys to what the function expects
res_xreg_remapped = {
    region: {
        "df_train": res_xreg_by_source[region]['data_monthly'],
        "df_test": res_xreg_by_source[region]['data_monthly_aug'],
    }
    for region in ALL_REGIONS
}

scaler_m, scaler_s, scaler_all = build_global_scalers_multi_source_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(f"blur_m={blur_m:.4f}  blur_s={blur_s:.4f}  blur_joint={blur_joint:.4f}")

#### Split glaciers of training sets:

In [ ]:
# --- temporarily add IT and AT as separate entries in res_xreg_by_source ---
res_xreg_by_source_ext = {
    **res_xreg_by_source,
    'IT': {
        'data_monthly':
        data_IT,
        'data_monthly_aug':
        res_xreg_by_source['IT_AT']['data_monthly_aug'][res_xreg_by_source[
            'IT_AT']['data_monthly_aug']['GLACIER'].isin(IT_GLACIERS)],
        'months_head_pad':
        res_xreg_by_source['IT_AT']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['IT_AT']['months_tail_pad'],
    },
    'AT': {
        'data_monthly':
        data_AT,
        'data_monthly_aug':
        res_xreg_by_source['IT_AT']['data_monthly_aug'][res_xreg_by_source[
            'IT_AT']['data_monthly_aug']['GLACIER'].isin(AT_GLACIERS)],
        'months_head_pad':
        res_xreg_by_source['IT_AT']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['IT_AT']['months_tail_pad'],
    },
}

RUN_GLACIER_RANK = False

for test_region, save_prefix in [("IT", "glacier_ranking_it"), ("AT", "glacier_ranking_at")]:
    rankings = {}
    for mode, topo_extra_cols in [
        ("joint", []),
        ("topo",  ["ELEVATION_DIFFERENCE"]),
    ]:
        save_path = BASE_DIR / f"{save_prefix}_{mode}.csv"

        if RUN_GLACIER_RANK:
            df_ranked = rank_glaciers_by_distance_to_target(
                res_xreg_by_source=res_xreg_by_source_ext,
                training_regions=ALL_REGIONS,
                test_region=test_region,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                blur_joint=blur_joint,
                seed=cfg.seed,
                mode=mode,
                topo_extra_cols=topo_extra_cols,
            )
            df_ranked.to_csv(save_path, index=False)
            print(f"Saved {test_region} {mode} ranking -> {save_path}")
        else:
            df_ranked = pd.read_csv(save_path)
            print(f"Loaded {test_region} {mode} ranking from {save_path}")

        rankings[mode] = df_ranked

    # store in named variables for downstream use
    if test_region == "IT":
        df_ranked_it_joint = rankings["joint"]
        df_ranked_it_topo  = rankings["topo"]
    else:
        df_ranked_at_joint = rankings["joint"]
        df_ranked_at_topo  = rankings["topo"]

    # comparison
    joint_top10 = set(rankings["joint"].head(10)['glacier'])
    topo_top10  = set(rankings["topo"].head(10)['glacier'])
    print(f"\n{test_region} — Top-10 overlap (joint ∩ topo): "
          f"{len(joint_top10 & topo_top10)} glaciers")
    print(f"  Joint only : {sorted(joint_top10 - topo_top10)}")
    print(f"  Topo only  : {sorted(topo_top10 - joint_top10)}")

## Train transformers:

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))

# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[0]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")
    
    
lstm_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

In [ ]:
FRACS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.4]
N_RANDOM_SEEDS = 10

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']

# --- 1. Build glacier subsets for all rankings ---
gl_subsets_itat = {}

for test_region, mode, df_ranked in [
    ("IT", "joint", df_ranked_it_joint),
    ("IT", "topo",  df_ranked_it_topo),
    ("AT", "joint", df_ranked_at_joint),
    ("AT", "topo",  df_ranked_at_topo),
]:
    key = f"{test_region}_{mode}"
    gl_subsets_itat[key] = build_glacier_subsets(
        df_ranked=df_ranked,
        fracs=FRACS,
        n_random_seeds=N_RANDOM_SEEDS,
        seed=cfg.seed,
    )
    print(f"\nSubsets built for {key}")

# convenience aliases
gl_subsets_it_joint = gl_subsets_itat["IT_joint"]
gl_subsets_it_topo  = gl_subsets_itat["IT_topo"]
gl_subsets_at_joint = gl_subsets_itat["AT_joint"]
gl_subsets_at_topo  = gl_subsets_itat["AT_topo"]

# --- 2. Dummy splits ---
dummy_splits = {
    region: {"holdout_glaciers": [], "pool_glaciers": []}
    for region in ALL_REGIONS
}

# --- 3. Test datasets ---
df_itat_full = res_xreg_by_source['IT_AT']['data_monthly']
df_itat_aug  = res_xreg_by_source['IT_AT']['data_monthly_aug']

ds_it = build_combined_LSTM_dataset(
    df_loss=df_itat_full[df_itat_full['GLACIER'].isin(IT_GLACIERS)],
    df_full=df_itat_aug[ df_itat_aug['GLACIER'].isin(IT_GLACIERS)],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['IT_AT']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['IT_AT']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)

ds_at = build_combined_LSTM_dataset(
    df_loss=df_itat_full[df_itat_full['GLACIER'].isin(AT_GLACIERS)],
    df_full=df_itat_aug[ df_itat_aug['GLACIER'].isin(AT_GLACIERS)],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['IT_AT']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['IT_AT']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)

print(f"IT only : {len(ds_it)} sequences")
print(f"AT only : {len(ds_at)} sequences")

df_itat_full = res_xreg_by_source['IT_AT']['data_monthly']
df_itat_aug = res_xreg_by_source['IT_AT']['data_monthly_aug']

ds_it = build_combined_LSTM_dataset(
    df_loss=df_itat_full[df_itat_full['GLACIER'].isin(IT_GLACIERS)],
    df_full=df_itat_aug[df_itat_aug['GLACIER'].isin(IT_GLACIERS)],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['IT_AT']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['IT_AT']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)

ds_at = build_combined_LSTM_dataset(
    df_loss=df_itat_full[df_itat_full['GLACIER'].isin(AT_GLACIERS)],
    df_full=df_itat_aug[df_itat_aug['GLACIER'].isin(AT_GLACIERS)],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['IT_AT']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['IT_AT']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)

print(f"IT only : {len(ds_it)} sequences")
print(f"AT only : {len(ds_at)} sequences")

In [ ]:
# --- 3. Train transformer and LSTM for each subset ---
TRAIN_FLAG = True
MODEL_DATE = "2026-05-21"

CACHE_DIR_RANK_ITAT = CACHE_DIR / "ranking_itat"
os.makedirs(CACHE_DIR_RANK_ITAT, exist_ok=True)

ranking_results_itat = []

SOURCE_REGIONS = [  # regions for "foundation model"
    'CH',
    'NOR',
    'ISL',
    'FR',
]

for ranking_label, df_ranked, gl_subsets_cur, ds_test_match, test_label_match in [
    ("ranked_by_IT_joint", df_ranked_it_joint, gl_subsets_it_joint, ds_it, "IT"),
    #("ranked_by_IT_topo",  df_ranked_it_topo,  gl_subsets_it_topo,  ds_it, "IT"),
    ("ranked_by_AT_joint", df_ranked_at_joint, gl_subsets_at_joint, ds_at, "AT"),
    #("ranked_by_AT_topo",  df_ranked_at_topo,  gl_subsets_at_topo,  ds_at, "AT"),
]:
    print(f"\n{'='*60}")
    print(f"  Ranking: {ranking_label}  →  eval on {test_label_match}")

    mode       = ranking_label.split("_")[-1]   # "joint" or "topo"
    models_dir = BASE_DIR / "models/ranking_itat" / mode
    os.makedirs(models_dir, exist_ok=True)

    for pct, subset in gl_subsets_cur.items():
        print(f"\n  Fraction: {pct}%")

        strategies = {
            "closest": subset['closest'],
            **{f"random_{s['seed_idx']}": s['glaciers']
               for s in subset['random']},
        }

        for strategy_name, glaciers in strategies.items():
            print(f"\n  --- {strategy_name} ({len(glaciers)} glaciers) ---")

            assets = build_assets_from_glacier_list(
                glaciers=glaciers,
                df_ranked=df_ranked,
                res_xreg_by_source=res_xreg_by_source,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                cfg=cfg,
                cache_path=str(
                    CACHE_DIR_RANK_ITAT /
                    f"assets_{test_label_match}_{mode}_{pct}pct_{strategy_name}.joblib"
                ),
                force_recompute=False,
                months_head_pad=months_head_pad,
                months_tail_pad=months_tail_pad,
                src_regions=SOURCE_REGIONS,
            )

            # --- train both model types ---
            for model_type, params, model_label in [
                ("transformer", best_params_gs, "transformer"),
                #("lstm",        lstm_params,    "lstm"),
            ]:
                model, model_path, info = train_or_load_one_source_model(
                    cfg=cfg,
                    key=f"{test_label_match}_{pct}pct_{strategy_name}",
                    lstm_assets=assets,
                    best_params=params,
                    device=device,
                    models_dir=models_dir,
                    prefix=f"{model_type}_rank_itat",
                    train_flag=TRAIN_FLAG,
                    force_retrain=False,
                    epochs=150,
                    date=MODEL_DATE,
                    model_type=model_type,
                    verbose=False,
                )

                ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    assets["ds_train"])
                ds_scaler.make_loaders(
                    train_idx=assets["train_idx"],
                    val_idx=assets["val_idx"],
                    fit_and_transform=True,
                    seed=cfg.seed,
                    verbose=False,
                )
                ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    ds_test_match)
                test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                    ds_copy, ds_scaler, batch_size=128, seed=cfg.seed,
                )
                metrics, _ = model.evaluate_with_preds(device, test_dl, ds_copy)

                is_random = strategy_name.startswith("random")
                print(f"    [{model_label}] [{test_label_match}] "
                      f"RMSE_a={metrics['RMSE_annual']:.3f}  "
                      f"RMSE_w={metrics['RMSE_winter']:.3f}  "
                      f"R2_a={metrics['R2_annual']:.3f}")

                ranking_results_itat.append({
                    "ranking_target": ranking_label,
                    "model":         model_label,
                    "pct":           pct,
                    "strategy":      "random" if is_random else strategy_name,
                    "seed_idx":      int(strategy_name.split("_")[1]) if is_random else None,
                    "test_region":   test_label_match,
                    "n_glaciers":    len(glaciers),
                    "n_sequences":   len(assets["ds_train"]),
                    **{k: round(v, 3) for k, v in metrics.items()},
                })

    # --- full baselines ---
    print(f"\n  Fraction: 100%")
    full_assets = build_assets_from_glacier_list(
        glaciers=df_ranked['glacier'].tolist(),
        df_ranked=df_ranked,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(
            CACHE_DIR_RANK_ITAT /
            f"assets_{test_label_match}_{mode}_100pct_full.joblib"
        ),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=SOURCE_REGIONS
    )

    for model_type, params, label in [
        ("lstm",        lstm_params,    "lstm_full"),
        ("transformer", best_params_gs, "transformer_full"),
    ]:
        model, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{test_label_match}_100pct_full",
            lstm_assets=full_assets,
            best_params=params,
            device=device,
            models_dir=models_dir,
            prefix=f"{model_type}_rank_itat",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=150,
            date=MODEL_DATE,
            model_type=model_type,
            verbose=False,
        )

        ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            full_assets["ds_train"])
        ds_scaler.make_loaders(
            train_idx=full_assets["train_idx"],
            val_idx=full_assets["val_idx"],
            fit_and_transform=True,
            seed=cfg.seed, verbose=False,
        )
        ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds_test_match)
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_copy, ds_scaler, batch_size=128, seed=cfg.seed,
        )
        metrics, _ = model.evaluate_with_preds(device, test_dl, ds_copy)

        print(f"  {label} [{test_label_match}]: "
              f"RMSE_a={metrics['RMSE_annual']:.3f}  "
              f"RMSE_w={metrics['RMSE_winter']:.3f}")

        ranking_results_itat.append({
            "ranking_target": ranking_label,
            "model":         label.replace("_full", ""),
            "pct":           100,
            "strategy":      label,
            "seed_idx":      None,
            "test_region":   test_label_match,
            "n_glaciers":    len(df_ranked),
            "n_sequences":   len(full_assets["ds_train"]),
            **{k: round(v, 3) for k, v in metrics.items()},
        })

df_ranking_results_itat = pd.DataFrame(ranking_results_itat)
df_ranking_results_itat.to_csv(BASE_DIR / "ranking_itat_results.csv", index=False)
print(df_ranking_results_itat.groupby(
    ["ranking_target", "model", "test_region", "pct", "strategy"]
).mean(numeric_only=True).round(3))

In [ ]:
# --- plots: 4 rankings x 2 models = 8 plots ---
for model_label in ["transformer"]:
    for ranking_label, test_region in [
        ("ranked_by_IT_joint", "IT"),
        #("ranked_by_IT_topo",  "IT"),
        ("ranked_by_AT_joint", "AT"),
        #("ranked_by_AT_topo",  "AT"),
    ]:
        plot_ranking_results_extended(
            df_results=df_ranking_results_itat,
            ranking_label=ranking_label,
            test_region=test_region,
            source_regions=ALL_REGIONS,
            n_rand_seeds=N_RANDOM_SEEDS,
            model_label=model_label,
            save_path=BASE_DIR / f"ranking_itat_{ranking_label}_{model_label}_eval_{test_region}",
        )

In [ ]:
FRACS_TO_PLOT = [10, 15, 100]

for ranking_label, df_ranked, gl_subsets_cur, ds_test, test_label in [
    ("ranked_by_IT_joint", df_ranked_it_joint, gl_subsets_it_joint, ds_it, "IT"),
    ("ranked_by_AT_joint", df_ranked_at_joint, gl_subsets_at_joint, ds_at, "AT"),
]:
    mode       = ranking_label.split("_")[-1]
    models_dir = BASE_DIR / "models/ranking_itat" / mode
    ranking_plot_configs = []

    for pct in FRACS_TO_PLOT:

        if pct == 100:
            # --- full pool (closest = all glaciers, no random) ---
            full_assets = build_assets_from_glacier_list(
                glaciers=df_ranked['glacier'].tolist(),
                df_ranked=df_ranked,
                res_xreg_by_source=res_xreg_by_source,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                cfg=cfg,
                cache_path=str(
                    CACHE_DIR_RANK_ITAT /
                    f"assets_{test_label}_{mode}_100pct_full.joblib"
                ),
                months_head_pad=months_head_pad,
                months_tail_pad=months_tail_pad,
                src_regions=SOURCE_REGIONS,
            )
            model_full, _, _ = train_or_load_one_source_model(
                cfg=cfg,
                key=f"{test_label}_100pct_full",
                lstm_assets=full_assets,
                best_params=best_params_gs,
                device=device,
                models_dir=models_dir,
                prefix="transformer_rank_itat",
                train_flag=False,
                date=MODEL_DATE,
                model_type="transformer",
                verbose=False,
            )
            ranking_plot_configs.append(
                (f"Full 100% ({mode})", model_full, full_assets))
            continue

        # --- closest ---
        assets_close = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['closest'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(
                CACHE_DIR_RANK_ITAT /
                f"assets_{test_label}_{mode}_{pct}pct_closest.joblib"
            ),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            src_regions=SOURCE_REGIONS,
        )
        model_close, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{test_label}_{pct}pct_closest",
            lstm_assets=assets_close,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix="transformer_rank_itat",
            train_flag=False,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        ranking_plot_configs.append(
            (f"Closest {pct}% ({mode})", model_close, assets_close))

        # --- random seed 0 ---
        assets_rnd = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['random'][0]['glaciers'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(
                CACHE_DIR_RANK_ITAT /
                f"assets_{test_label}_{mode}_{pct}pct_random_0.joblib"
            ),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            src_regions=SOURCE_REGIONS,
        )
        model_rnd, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{test_label}_{pct}pct_random_0",
            lstm_assets=assets_rnd,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix="transformer_rank_itat",
            train_flag=False,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        ranking_plot_configs.append(
            (f"Random {pct}% seed 0 ({mode})", model_rnd, assets_rnd))

    plot_pred_vs_truth_grid(
        plot_configs=ranking_plot_configs,
        ds_test=ds_test,
        device=device,
        cfg=cfg,
        ncols=3,   # closest | random | (empty for 100% row since only one panel)
        ax_xlim=(-8, 6),
        ax_ylim=(-8, 6),
        figsize=(25, 12),
        title=f"Closest vs random ({ranking_label}) → {test_label}",
        save_path=BASE_DIR / f"figures/ranking_itat_{ranking_label}_pred_vs_truth_{test_label}",
    )